# Patient-grouped nested cross-validation and clustered bootstrap

This notebook has been revised to address repeated admissions from the same patient.

- `subject_reference` is used **only as a grouping variable**, never as a model feature.
- Both the **outer cross-validation** and the **inner OOF cross-validation used for threshold selection** use `StratifiedGroupKFold`, so all admissions from one patient remain in the same fold.
- Explicit leakage checks verify that the number of overlapping patients between training and validation is zero in every outer and inner split.
- Bootstrap confidence intervals use a **patient-level cluster bootstrap**: patients are resampled with replacement and all admissions belonging to a sampled patient are included together.
- Cohort-level summaries report unique patients and the distribution of admissions per patient.

The performance estimand remains admission-level; clustering changes partitioning and uncertainty estimation to account for repeated admissions.


In [ ]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Iterable, Optional

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss,
)
from sklearn.linear_model import LogisticRegression

from tabpfn_extensions import TabPFNClassifier, interpretability


warnings.filterwarnings("ignore")

data_fs_static_external = pd.read_csv("YOUR_PATH")
data_fs_static_train = pd.read_csv("YOUR_PATH")

# Patient identifier used for grouped cross-validation and clustered bootstrap CIs.
# IMPORTANT: do not include this identifier in feature_space.
GROUP_COL = "subject_reference"

scale = 'yes'
feature_space = ['feature_1', 'feature_2', ...]

X_train, y_train  = do_train_test_split(data_fs_static_train,feature_space,scale)


In [ ]:
OUTER_SPLITS = 5
INNER_SPLITS_FOR_THRESHOLD = 5  # grouped OOF splits for leakage-free threshold selection within each outer-train
N_NESTED_TRIALS = 20
RANDOM_SEEDS = [1000 + i for i in range(N_NESTED_TRIALS)]

N_BOOTSTRAPS = 2000
BOOT_ALPHA = 0.05

DEVICE = "auto"  # "cpu" | "cuda" | "auto"

# SHAP-like permutation interpretability (TabPFN extensions)
RUN_SHAP = True
MAX_SHAP_SAMPLES_PER_FOLD = 1000  # to control runtime; set None to use all outer-test rows

OUT_DIR = "outputs_nested_tabpfn"
FIG_DIR = os.path.join(OUT_DIR, "figures")  # kept for symmetry; not used unless you add plots
SHAP_DIR = os.path.join(OUT_DIR, "shap_values")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)

# HF token pass-through if present
HF_TOKEN = os.environ.get("HF_TOKEN", None)
if HF_TOKEN is None:
    print("Note: HF_TOKEN not found in environment. If TabPFN requires it, set it before running.")
else:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("Ready.")


# ============================
# Helpers
# ============================

def as_numpy(y: Iterable) -> np.ndarray:
    return np.asarray(y).reshape(-1)

def safe_confusion(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[int, int, int, int]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return int(tn), int(fp), int(fn), int(tp)

def stable_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    return float(auc(rec, prec))

def point_metrics(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    tn, fp, fn, tp = safe_confusion(y_true, y_pred)
    spec = tn / (tn + fp + 1e-12)
    sens = tp / (tp + fn + 1e-12)
    npv  = tn / (tn + fn + 1e-12)

    out = {
        "auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float("nan"),
        "auprc": stable_auprc(y_true, y_prob),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(spec),
        "sensitivity": float(sens),
        "npv": float(npv),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "tn": float(tn), "fp": float(fp), "fn": float(fn), "tp": float(tp),
    }
    return out

def thresholds_from_predictions(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, float]:
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    if len(thresh) == 0:
        return {"Default-0.5": 0.5, "MCC-optimal": 0.5, "Youden": 0.5}

    t_default = 0.5

    # MCC-optimal
    mcc_vals = [matthews_corrcoef(y_true, (y_prob >= t).astype(int)) for t in thresh]
    t_mcc = float(thresh[int(np.nanargmax(mcc_vals))])

    # Youden J
    youden_vals = []
    for t in thresh:
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = safe_confusion(y_true, yp)
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        youden_vals.append(sens + spec - 1)
    t_youden = float(thresh[int(np.nanargmax(youden_vals))])

    return {"Default-0.5": t_default, "MCC-optimal": t_mcc, "Youden": t_youden}

def _logit(p: np.ndarray) -> np.ndarray:
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def calibration_in_the_large(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    z = _logit(y_prob).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.intercept_[0])

def calibration_slope(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    z = _logit(y_prob).reshape(-1, 1)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=2000)
    lr.fit(z, y_true)
    return float(lr.coef_[0][0])


# ============================
# TabPFN OOF probabilities (for leakage-free threshold selection)
# ============================

def get_oof_probabilities_tabpfn(
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    device: str,
    n_splits: int,
    seed: int,
) -> np.ndarray:
    """
    Grouped OOF probabilities for TabPFN on the outer-training dataset.

    All admissions belonging to the same patient are kept in the same inner fold.
    These OOF predictions are used only for leakage-free threshold selection.
    """
    y = as_numpy(y)
    groups = as_numpy(groups)

    if len(X) != len(y) or len(y) != len(groups):
        raise ValueError("X, y, and groups must have identical lengths.")

    oof = np.full(shape=(len(X),), fill_value=np.nan, dtype=float)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for tr_idx, va_idx in cv.split(X, y, groups=groups):
        tr_groups = set(groups[tr_idx].tolist())
        va_groups = set(groups[va_idx].tolist())
        overlap = tr_groups.intersection(va_groups)
        if overlap:
            raise RuntimeError(
                f"Patient leakage detected in inner CV: {len(overlap)} patient(s) occur in both folds."
            )

        X_tr = X.iloc[tr_idx].to_numpy()
        y_tr = y[tr_idx]
        X_va = X.iloc[va_idx].to_numpy()

        m = TabPFNClassifier(device=device)
        m.fit(X_tr, y_tr)
        oof[va_idx] = m.predict_proba(X_va)[:, 1]

    if np.isnan(oof).any():
        raise RuntimeError("OOF probabilities contain NaNs; check grouped CV/data.")
    return oof


# ============================
# Cluster bootstrap (patient-level resampling)
# ============================

def _prepare_cluster_indices(groups: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray]]:
    """
    Precompute row indices for each patient/cluster.
    """
    groups = as_numpy(groups)
    if pd.isna(groups).any():
        raise ValueError("Patient identifiers contain missing values; grouped resampling requires complete IDs.")

    unique_groups = pd.unique(groups)
    rows_by_group = [np.flatnonzero(groups == g) for g in unique_groups]
    return np.asarray(unique_groups, dtype=object), rows_by_group


def cluster_bootstrap_indices(
    rng: np.random.Generator,
    rows_by_group: List[np.ndarray],
) -> np.ndarray:
    """
    Resample patients with replacement and include ALL admissions for every sampled patient.

    If a patient is sampled more than once, all of that patient's admissions are repeated,
    as required for a standard nonparametric cluster bootstrap.
    """
    n_groups = len(rows_by_group)
    sampled_group_positions = rng.integers(0, n_groups, size=n_groups, endpoint=False)
    return np.concatenate([rows_by_group[j] for j in sampled_group_positions])


def bootstrap_metric_distribution(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    y_pred: np.ndarray,
    groups: np.ndarray,
    metrics: List[str],
    n_boot: int,
    seed: int,
) -> pd.DataFrame:
    """
    Admission-level performance metrics with patient-clustered bootstrap CIs.
    The estimand remains admission-level; only the uncertainty calculation is clustered.
    """
    y_true = as_numpy(y_true)
    y_prob = as_numpy(y_prob)
    y_pred = as_numpy(y_pred)
    groups = as_numpy(groups)

    if not (len(y_true) == len(y_prob) == len(y_pred) == len(groups)):
        raise ValueError("y_true, y_prob, y_pred, and groups must have identical lengths.")

    rng = np.random.default_rng(seed)
    _, rows_by_group = _prepare_cluster_indices(groups)

    out_rows = []
    for b in range(n_boot):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        m = point_metrics(y_true[idx], y_prob[idx], y_pred[idx])
        for metric in metrics:
            out_rows.append({"boot_id": b, "metric": metric, "boot_value": float(m[metric])})
    return pd.DataFrame(out_rows)


# ============================
# SHAP-like (TabPFN permutation) helpers
# ============================

def _subsample_rows(X: np.ndarray, n: Optional[int], seed: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns (X_sub, idx_sub_in_original).
    If n is None or len(X)<=n, returns all.
    """
    if n is None or len(X) <= n:
        return X, np.arange(len(X))
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), n, replace=False)
    return X[idx], idx

def tabpfn_permutation_shap_values(
    estimator: TabPFNClassifier,
    test_x: np.ndarray,
    feature_names: List[str],
) -> np.ndarray:
    """
    Wrapper around tabpfn_extensions interpretability permutation SHAP.
    Returns array (n_samples, n_features) for the positive class when possible.
    """
    S = interpretability.shap.get_shap_values(
        estimator=estimator,
        test_x=test_x,
        attribute_names=feature_names,
        algorithm="permutation",
    )
    arr = np.asarray(S)

    # Standardize shapes:
    # Possible: (2, n, p) or (n, p) or (n, p, 2)
    if arr.ndim == 3 and arr.shape[0] == 2:
        return arr[1]
    if arr.ndim == 3 and arr.shape[-1] == 2:
        return arr[..., 1]
    if arr.ndim == 2:
        return arr
    raise ValueError(f"Unexpected SHAP array shape: {arr.shape}")


# ============================
# Nested CV runner
# ============================

@dataclass
class NestedCVResult:
    seed: int
    y_true: np.ndarray
    p: np.ndarray
    idx: np.ndarray
    groups: np.ndarray
    fold_id: np.ndarray
    y_pred_default: np.ndarray
    y_pred_mcc: np.ndarray
    y_pred_youden: np.ndarray
    thresholds_per_fold: List[Dict[str, float]]
    shap_values: Optional[np.ndarray]       # pooled (n, p) over outer-test subsamples, or None
    shap_row_index: Optional[np.ndarray]    # indices aligned to shap_values rows (original X index), or None
    feature_names: List[str]
    fold_diagnostics: pd.DataFrame

def run_nested_cv_once_tabpfn(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    outer_splits: int,
    n_splits_threshold: int,
    seed: int,
    device: str,
    run_shap: bool,
    max_shap_samples_per_fold: Optional[int],
) -> NestedCVResult:
    y_np = as_numpy(y)
    groups_np = as_numpy(groups)

    if len(X) != len(y_np) or len(y_np) != len(groups_np):
        raise ValueError("X, y, and groups must have identical lengths.")
    if pd.isna(groups_np).any():
        raise ValueError("subject_reference contains missing values.")

    outer_cv = StratifiedGroupKFold(n_splits=outer_splits, shuffle=True, random_state=seed)

    y_true_all, p_all, idx_all, groups_all, fold_id_all = [], [], [], [], []
    pred_default_all, pred_mcc_all, pred_youden_all = [], [], []
    thresholds_per_fold: List[Dict[str, float]] = []

    feature_names = list(X.columns.astype(str))

    shap_blocks = []
    shap_idx_blocks = []
    fold_diagnostics = []

    for fold, (tr_idx, te_idx) in enumerate(
        outer_cv.split(X, y_np, groups=groups_np), start=1
    ):
        X_tr_df, y_tr = X.iloc[tr_idx], y_np[tr_idx]
        X_te_df, y_te = X.iloc[te_idx], y_np[te_idx]
        groups_tr = groups_np[tr_idx]
        groups_te = groups_np[te_idx]

        # Explicitly verify that no patient occurs in both outer train and validation.
        train_patients = set(groups_tr.tolist())
        test_patients = set(groups_te.tolist())
        overlap = train_patients.intersection(test_patients)
        if overlap:
            raise RuntimeError(
                f"Patient leakage detected in outer fold {fold}: "
                f"{len(overlap)} patient(s) occur in both train and validation."
            )

        fold_diagnostics.append({
            "seed": seed,
            "fold": fold,
            "n_train_admissions": len(tr_idx),
            "n_validation_admissions": len(te_idx),
            "n_train_patients": len(train_patients),
            "n_validation_patients": len(test_patients),
            "n_overlapping_patients": len(overlap),
        })

        # --- 1) leakage-free threshold selection from OOF on outer-train
        p_tr_oof = get_oof_probabilities_tabpfn(
            X=X_tr_df,
            y=y_tr,
            groups=groups_tr,
            device=device,
            n_splits=n_splits_threshold,
            seed=seed,
        )
        thr = thresholds_from_predictions(y_tr, p_tr_oof)
        thresholds_per_fold.append(thr)

        # --- 2) fit model on full outer-train, predict outer-test
        model = TabPFNClassifier(device=device)
        model.fit(X_tr_df.to_numpy(), y_tr)
        p_te = model.predict_proba(X_te_df.to_numpy())[:, 1]

        y_pred_default = (p_te >= thr["Default-0.5"]).astype(int)
        y_pred_mcc     = (p_te >= thr["MCC-optimal"]).astype(int)
        y_pred_youden  = (p_te >= thr["Youden"]).astype(int)

        y_true_all.append(y_te)
        p_all.append(p_te)
        idx_all.append(X_te_df.index.to_numpy())
        groups_all.append(groups_te)
        fold_id_all.append(np.full(len(te_idx), fold, dtype=int))

        pred_default_all.append(y_pred_default)
        pred_mcc_all.append(y_pred_mcc)
        pred_youden_all.append(y_pred_youden)

        # --- 3) SHAP-like permutation on outer-test (subsampled), pooled across folds
        if run_shap:
            X_te_np = X_te_df.to_numpy(dtype=np.float64)
            X_sub, sub_idx = _subsample_rows(X_te_np, max_shap_samples_per_fold, seed=seed + fold)
            # Keep original index labels aligned
            idx_labels = X_te_df.index.to_numpy()[sub_idx]

            # IMPORTANT: Use the already-fitted model for SHAP
            S = tabpfn_permutation_shap_values(
                estimator=model,
                test_x=X_sub,
                feature_names=feature_names,
            )
            shap_blocks.append(S)
            shap_idx_blocks.append(idx_labels)

    shap_values = None
    shap_row_index = None
    if run_shap and len(shap_blocks) > 0:
        shap_values = np.vstack(shap_blocks)
        shap_row_index = np.concatenate(shap_idx_blocks)

    return NestedCVResult(
        seed=seed,
        y_true=np.concatenate(y_true_all),
        p=np.concatenate(p_all),
        idx=np.concatenate(idx_all),
        groups=np.concatenate(groups_all),
        fold_id=np.concatenate(fold_id_all),
        y_pred_default=np.concatenate(pred_default_all),
        y_pred_mcc=np.concatenate(pred_mcc_all),
        y_pred_youden=np.concatenate(pred_youden_all),
        thresholds_per_fold=thresholds_per_fold,
        shap_values=shap_values,
        shap_row_index=shap_row_index,
        feature_names=feature_names,
        fold_diagnostics=pd.DataFrame(fold_diagnostics),
    )


# ============================
# Patient/admission summaries + group alignment
# ============================

def summarize_admissions_per_patient(
    df: pd.DataFrame,
    cohort_name: str,
    group_col: str = GROUP_COL,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Assumes one row in df corresponds to one modeled hospital admission/stay.
    Produces reviewer-ready cohort summaries and the full admissions-per-patient distribution.
    """
    if group_col not in df.columns:
        raise KeyError(f"{group_col!r} is not present in cohort {cohort_name!r}.")
    if df[group_col].isna().any():
        raise ValueError(f"{cohort_name}: {group_col} contains missing values.")

    counts = df.groupby(group_col, dropna=False).size().rename("n_admissions")
    q = counts.quantile([0.25, 0.50, 0.75])

    summary = pd.DataFrame([{
        "cohort": cohort_name,
        "n_admissions": int(len(df)),
        "n_unique_patients": int(counts.size),
        "n_patients_with_recurrent_admissions": int((counts > 1).sum()),
        "pct_patients_with_recurrent_admissions": float(100 * (counts > 1).mean()),
        "recurrent_admissions_present": bool((counts > 1).any()),
        "admissions_per_patient_mean": float(counts.mean()),
        "admissions_per_patient_sd": float(counts.std(ddof=1)) if counts.size > 1 else 0.0,
        "admissions_per_patient_min": int(counts.min()),
        "admissions_per_patient_q1": float(q.loc[0.25]),
        "admissions_per_patient_median": float(q.loc[0.50]),
        "admissions_per_patient_q3": float(q.loc[0.75]),
        "admissions_per_patient_max": int(counts.max()),
    }])

    distribution = (
        counts.value_counts()
        .sort_index()
        .rename_axis("n_admissions")
        .rename("n_patients")
        .reset_index()
    )
    distribution.insert(0, "cohort", cohort_name)
    return summary, distribution


def align_groups_to_model_rows(
    source_df: pd.DataFrame,
    X: pd.DataFrame,
    group_col: str = GROUP_COL,
) -> pd.Series:
    """
    Align patient IDs from the source cohort to the rows returned by preprocessing.

    Preferred path: preserve the source DataFrame index through do_train_test_split().
    A same-length/order fallback is allowed, but a warning is emitted.
    """
    if group_col not in source_df.columns:
        raise KeyError(f"{group_col!r} not found in data_fs_static_train.")
    if source_df[group_col].isna().any():
        raise ValueError(f"{group_col} contains missing values.")

    # Safest case: preprocessing retained exactly the same row index.
    if X.index.equals(source_df.index):
        groups = source_df.loc[X.index, group_col].copy()
        groups.index = X.index
        return groups

    # A non-trivial subset index can also be safely mapped if original index labels were preserved.
    # Avoid silently accepting RangeIndex(0..n-1) after row filtering/resetting, which is ambiguous.
    ambiguous_reset_index = (
        len(X) < len(source_df)
        and isinstance(X.index, pd.RangeIndex)
        and X.index.start == 0
        and X.index.step == 1
    )
    if (
        not ambiguous_reset_index
        and source_df.index.is_unique
        and X.index.isin(source_df.index).all()
    ):
        groups = source_df.loc[X.index, group_col].copy()
        groups.index = X.index
        return groups

    # Fallback if preprocessing preserved row order but reset/replaced the index.
    if len(source_df) == len(X):
        warnings.warn(
            "X_train does not retain a directly mappable source index. "
            "Assuming preprocessing preserved row order when aligning subject_reference. "
            "For maximum safety, preserve the original DataFrame index in do_train_test_split()."
        )
        return pd.Series(source_df[group_col].to_numpy(), index=X.index, name=group_col)

    raise ValueError(
        "Could not safely align subject_reference to X_train. "
        "Preserve the original row index through do_train_test_split(), or return the patient IDs alongside X/y."
    )


# Reviewer-requested cohort-level admission summaries.
cohort_summary_frames = []
cohort_distribution_frames = []
for _cohort_name, _df in [
    ("internal", data_fs_static_train),
    ("external", data_fs_static_external),
]:
    if GROUP_COL in _df.columns:
        _summary, _distribution = summarize_admissions_per_patient(_df, _cohort_name)
        cohort_summary_frames.append(_summary)
        cohort_distribution_frames.append(_distribution)
    else:
        print(f"Skipping patient/admission summary for {_cohort_name}: {GROUP_COL!r} not found.")

if cohort_summary_frames:
    cohort_patient_summary_df = pd.concat(cohort_summary_frames, ignore_index=True)
    cohort_patient_summary_path = os.path.join(OUT_DIR, "cohort_patient_admission_summary.csv")
    cohort_patient_summary_df.to_csv(cohort_patient_summary_path, index=False)
    print(f"Saved cohort patient/admission summary: {cohort_patient_summary_path}")
    print(cohort_patient_summary_df)

if cohort_distribution_frames:
    admissions_per_patient_distribution_df = pd.concat(cohort_distribution_frames, ignore_index=True)
    admissions_per_patient_distribution_path = os.path.join(OUT_DIR, "admissions_per_patient_distribution.csv")
    admissions_per_patient_distribution_df.to_csv(admissions_per_patient_distribution_path, index=False)
    print(f"Saved admissions-per-patient distribution: {admissions_per_patient_distribution_path}")


# ============================
# Run
# ============================

if not isinstance(y_train, pd.Series):
    y_train = pd.Series(y_train, index=X_train.index)

X = X_train.copy()
y = y_train.copy()
groups = align_groups_to_model_rows(data_fs_static_train, X, GROUP_COL)

# subject_reference must never be a model feature.
if GROUP_COL in X.columns:
    raise ValueError(
        f"{GROUP_COL} is present in X. Remove patient identifiers from feature_space before modeling."
    )

if not (X.index.equals(y.index) and X.index.equals(groups.index)):
    y = y.reindex(X.index)
    groups = groups.reindex(X.index)

if y.isna().any() or groups.isna().any():
    raise ValueError("Could not align X, y, and subject_reference without missing values.")

print(
    f"Internal modeling cohort: {len(X)} admissions from "
    f"{groups.nunique()} unique patients; "
    f"{(groups.value_counts() > 1).sum()} patients have recurrent admissions."
)

all_trial_preds: List[NestedCVResult] = []

print(f"Running repeated nested CV (TabPFN) ({N_NESTED_TRIALS} trials)...")
for i, seed in enumerate(RANDOM_SEEDS, start=1):
    print(f"\n=== Trial {i}/{len(RANDOM_SEEDS)} | seed={seed} ===")
    res = run_nested_cv_once_tabpfn(
        X=X,
        y=y,
        groups=groups,
        outer_splits=OUTER_SPLITS,
        n_splits_threshold=INNER_SPLITS_FOR_THRESHOLD,
        seed=seed,
        device=DEVICE,
        run_shap=RUN_SHAP,
        max_shap_samples_per_fold=MAX_SHAP_SAMPLES_PER_FOLD,
    )
    all_trial_preds.append(res)

    # One patient must map to exactly one validation fold within each repeated-CV trial.
    patient_fold_counts = (
        pd.DataFrame({"patient": res.groups, "fold": res.fold_id})
        .groupby("patient")["fold"]
        .nunique()
    )
    if int(patient_fold_counts.max()) != 1:
        raise RuntimeError("A patient was assigned to more than one outer validation fold.")

print("\nNested CV runs complete.")

# Save explicit fold-level leakage diagnostics for the reviewer response.
cv_group_diagnostics_df = pd.concat(
    [r.fold_diagnostics.assign(trial=i) for i, r in enumerate(all_trial_preds, start=1)],
    ignore_index=True,
)
cv_group_diagnostics_path = os.path.join(OUT_DIR, "grouped_cv_fold_diagnostics.csv")
cv_group_diagnostics_df.to_csv(cv_group_diagnostics_path, index=False)
print(f"Saved grouped-CV diagnostics: {cv_group_diagnostics_path}")
print(
    "Maximum number of overlapping patients in any outer fold:",
    int(cv_group_diagnostics_df["n_overlapping_patients"].max()),
)


# ============================
# Metrics + BOOTSTRAP CIs
# ============================

METRICS_TO_REPORT = [
    "auc", "auprc", "f1", "mcc",
    "accuracy", "precision", "recall", "specificity", "sensitivity", "npv",
    "brier"
]
RULES = ["Default-0.5", "MCC-optimal", "Youden"]

# Per-trial point estimates
trial_point_rows = []
for t, res in enumerate(all_trial_preds, start=1):
    rule_to_pred = {
        "Default-0.5": res.y_pred_default,
        "MCC-optimal": res.y_pred_mcc,
        "Youden": res.y_pred_youden,
    }
    for rule, y_pred in rule_to_pred.items():
        m = point_metrics(res.y_true, res.p, y_pred)
        trial_point_rows.append({"trial": t, "seed": res.seed, "rule": rule,
                                 **{k: m[k] for k in METRICS_TO_REPORT}})

trial_point_df = pd.DataFrame(trial_point_rows)
trial_point_path = os.path.join(OUT_DIR, "nested_tabpfn_trial_point_metrics.csv")
trial_point_df.to_csv(trial_point_path, index=False)
print(f"Saved per-trial point estimates: {trial_point_path}")

# Patient-clustered bootstrap within each trial, pooled across trials per rule+metric
all_boot_rows = []
for t, res in enumerate(all_trial_preds, start=1):
    rule_to_pred = {
        "Default-0.5": res.y_pred_default,
        "MCC-optimal": res.y_pred_mcc,
        "Youden": res.y_pred_youden,
    }

    stable_rule_seed_offset = {
        "Default-0.5": 101,
        "MCC-optimal": 202,
        "Youden": 303,
    }
    for rule, y_pred in rule_to_pred.items():
        boot_seed = int(res.seed + stable_rule_seed_offset[rule])
        dist = bootstrap_metric_distribution(
            y_true=res.y_true,
            y_prob=res.p,
            y_pred=y_pred,
            groups=res.groups,
            metrics=METRICS_TO_REPORT,
            n_boot=N_BOOTSTRAPS,
            seed=boot_seed,
        )
        dist["trial"] = t
        dist["seed"] = res.seed
        dist["rule"] = rule
        all_boot_rows.append(dist)

boot_df = pd.concat(all_boot_rows, ignore_index=True)
boot_path = os.path.join(OUT_DIR, f"nested_tabpfn_cluster_bootstrap_distributions_{N_BOOTSTRAPS}x{N_NESTED_TRIALS}.csv")
boot_df.to_csv(boot_path, index=False)
print(f"Saved bootstrap distributions (may be large): {boot_path}")

# Summarize pooled bootstrap distributions per rule+metric
summary_rows = []
for rule in RULES:
    for metric in METRICS_TO_REPORT:
        g = boot_df[(boot_df["rule"] == rule) & (boot_df["metric"] == metric)]["boot_value"].to_numpy(dtype=float)

        ci_low = float(np.nanpercentile(g, 100 * (BOOT_ALPHA / 2)))
        ci_high = float(np.nanpercentile(g, 100 * (1 - BOOT_ALPHA / 2)))

        pe = float(trial_point_df[trial_point_df["rule"] == rule][metric].mean())

        summary_rows.append({
            "rule": rule,
            "metric": metric,
            "point_estimate_mean_over_trials": pe,
            "bootstrap_ci_low": ci_low,
            "bootstrap_ci_high": ci_high,
            "n_boot_total": int(len(g)),
        })

bootstrap_summary_df = pd.DataFrame(summary_rows)
bootstrap_summary_path = os.path.join(OUT_DIR, "nested_tabpfn_cluster_bootstrap_CI_summary.csv")
bootstrap_summary_df.to_csv(bootstrap_summary_path, index=False)
print(f"Saved bootstrap CI summary: {bootstrap_summary_path}")
print(bootstrap_summary_df)


# ============================
# Calibration per trial + bootstrap pooled across trials
# ============================

cal_point_rows = []
cal_boot_rows = []

for t, res in enumerate(all_trial_preds, start=1):
    citl = calibration_in_the_large(res.y_true, res.p)
    slope = calibration_slope(res.y_true, res.p)
    brier = float(brier_score_loss(res.y_true, res.p))
    cal_point_rows.append({"trial": t, "seed": res.seed, "brier": brier, "citl": citl, "slope": slope})

    rng = np.random.default_rng(res.seed + 99_999)
    _, rows_by_group = _prepare_cluster_indices(res.groups)
    for b in range(N_BOOTSTRAPS):
        idx = cluster_bootstrap_indices(rng, rows_by_group)
        yt = res.y_true[idx]
        pr = res.p[idx]

        cal_boot_rows.append({"trial": t, "seed": res.seed, "boot_id": b, "metric": "brier",
                              "boot_value": float(brier_score_loss(yt, pr))})

        # Extremely rare bootstrap samples can contain only one outcome class;
        # calibration intercept/slope are undefined for those replicates.
        if len(np.unique(yt)) < 2:
            citl_b = np.nan
            slope_b = np.nan
        else:
            try:
                citl_b = float(calibration_in_the_large(yt, pr))
                slope_b = float(calibration_slope(yt, pr))
            except Exception:
                citl_b = np.nan
                slope_b = np.nan

        cal_boot_rows.append({"trial": t, "seed": res.seed, "boot_id": b, "metric": "citl",
                              "boot_value": citl_b})
        cal_boot_rows.append({"trial": t, "seed": res.seed, "boot_id": b, "metric": "slope",
                              "boot_value": slope_b})

cal_point_df = pd.DataFrame(cal_point_rows)
cal_point_path = os.path.join(OUT_DIR, "nested_tabpfn_calibration_trial_points.csv")
cal_point_df.to_csv(cal_point_path, index=False)

cal_boot_df = pd.DataFrame(cal_boot_rows)
cal_boot_path = os.path.join(OUT_DIR, f"nested_tabpfn_calibration_cluster_bootstrap_{N_BOOTSTRAPS}x{N_NESTED_TRIALS}.csv")
cal_boot_df.to_csv(cal_boot_path, index=False)

cal_summary_rows = []
for metric in ["brier", "citl", "slope"]:
    g = cal_boot_df[cal_boot_df["metric"] == metric]["boot_value"].to_numpy(dtype=float)
    ci_low = float(np.nanpercentile(g, 100 * (BOOT_ALPHA / 2)))
    ci_high = float(np.nanpercentile(g, 100 * (1 - BOOT_ALPHA / 2)))
    pe = float(cal_point_df[metric].mean())
    cal_summary_rows.append({
        "metric": metric,
        "point_estimate_mean_over_trials": pe,
        "bootstrap_ci_low": ci_low,
        "bootstrap_ci_high": ci_high,
        "n_boot_total": int(len(g)),
    })

cal_summary_df = pd.DataFrame(cal_summary_rows)
cal_summary_path = os.path.join(OUT_DIR, "nested_tabpfn_calibration_cluster_bootstrap_CI_summary.csv")
cal_summary_df.to_csv(cal_summary_path, index=False)

print(f"Saved calibration point estimates: {cal_point_path}")
print(f"Saved calibration bootstrap distributions: {cal_boot_path}")
print(f"Saved calibration bootstrap CI summary: {cal_summary_path}")
print(cal_summary_df)


# ============================
# SHAP-like aggregation across ALL trials (pooled outer-test subsamples)
# ============================

if RUN_SHAP:
    shap_blocks = [r.shap_values for r in all_trial_preds if r.shap_values is not None]
    if len(shap_blocks) == 0:
        print("\nNo SHAP values were collected (RUN_SHAP=True but results empty).")
    else:
        shap_stack = np.vstack(shap_blocks)  # (sum_n_subsampled, n_features)
        feature_names = all_trial_preds[0].feature_names

        shap_mean = shap_stack.mean(axis=0)
        shap_mean_abs = np.abs(shap_stack).mean(axis=0)

        shap_summary_df = pd.DataFrame({
            "feature": feature_names,
            "mean_shap": shap_mean,
            "mean_abs_shap": shap_mean_abs,
        }).sort_values("mean_abs_shap", ascending=False)

        shap_path = os.path.join(SHAP_DIR, "nested_tabpfn_permshap_aggregated_over_20_trials.csv")
        shap_summary_df.to_csv(shap_path, index=False)

        print(f"\nSaved aggregated TabPFN permutation-SHAP summary: {shap_path}")
        print(shap_summary_df.head(25))

print("\nDone.")
